# NÃO RODAR

In [10]:
import pandas as pd
import re

In [ ]:
#pega o poemas_com_obras2.csv, e tira todos os com  titulo vazio
df = pd.read_csv('poemas_com_obras2.csv')

# reescrever a coluna ['id'], com poema_001, poema_002, etc
df['id'] = ['poema_{:03d}'.format(i) for i in range(1, len(df) + 1)]

df.dropna(subset=['titulo'], inplace=True)

# tirar a primeira ocorrencia de "/" em cada linha, da coluna ['estrofes']
df['estrofes'] = df['estrofes'].str.replace('/', '', n=1)

df.to_csv('poemas_com_obras.csv', index=False)

In [9]:
# Juntar 2 arquivos csv (poemas.csv e poemas_com_obras.csv), priorizando o poemas_com_obras.csv e juntando pelo numero na coluna 'página'
# 1. Carregar os dois arquivos CSV
df_poemas = pd.read_csv('poemas.csv')
df_obras = pd.read_csv('poemas_com_obras.csv')

# 2. Juntar os dois dataframes colocando o arquivo com obras PRIMEIRO
df_combinado = pd.concat([df_obras, df_poemas])

# 3. Remover as duplicatas da coluna 'pagina', mantendo a primeira ocorrência (que veio do df_obras)
df_combinado = df_combinado.drop_duplicates(subset=['pagina'], keep='first')

# 4. Ordenar pela página para manter a sequência correta do livro
df_combinado = df_combinado.sort_values(by='pagina')

# 5. Salvar por cima do poemas.csv original
df_combinado.to_csv('poemas.csv', index=False, encoding='utf-8')

#mostrar linha com 'pagina' = 43
df_combinado[df_combinado['pagina'] == 43]

,id,pagina,estrofes,titulo,ano,tecnica,local,dimensoes,colecao
10,poema_014,43,Galopei o vento e também / Tornei-me invisív...,Noite em Brodowski,1955,Desenho a crayon colorido e grafite/papel,Local desconhecido,"20,5 × 27,5 cm",Coleção particular


### Ajustando o csv completo

In [11]:
df_full = pd.read_csv('poemas.csv')

# reescrever a coluna ['id'], com poema_001, poema_002, etc
df_full['id'] = ['poema_{:03d}'.format(i) for i in range(1, len(df_full) + 1)]

df_full['estrofes'] = (
    df_full['estrofes']
    .str.replace(r'\s*/?\s*\d{2,3}\s*$', '', regex=True)
    .str.rstrip()
)

# Para estrair local e data de poemas

PATTERN = r'(?:(?:De\s+\w+\s+a\s+)?([A-ZÀ-Ú][A-Za-záéíóúãõâêîôûç\s]+?),\s*)?(\d{1,2}[º°]?\s*)?(\w+\.),?\s*(\d{4})\.?'

def extrair_localdata(estrofe):
    # Pegar só o final (últimos 60 chars) pra evitar falsos positivos
    tail = str(estrofe)[-60:]
    m = re.search(PATTERN, tail)
    if not m:
        return None, None, None, None
    return m.group(1), m.group(2), m.group(3), m.group(4)

def remover_localdata(estrofe):
    return re.sub(
        r'\s*/?\s*(?:(?:De\s+\w+\s+a\s+)?[A-ZÀ-Ú][A-Za-záéíóúãõâêîôûç\s]+?,\s*)?\d{0,2}[º°]?\s*\w+\.,?\s*\d{4}\.?\s*$',
        '', str(estrofe)
    ).rstrip()

# Extrair para novas colunas
df_full[['local_poema', 'dia_poema', 'mes_poema', 'ano_poema']] = df_full['estrofes'].apply(
    lambda x: pd.Series(extrair_localdata(x))
)

# Remover da string original
df_full['estrofes'] = df_full['estrofes'].apply(remover_localdata)

In [13]:
# salvar por cima do poemas.csv original
df_full.to_csv('poemas.csv', index=False, encoding='utf-8')